# <font size=7><strong>Transcribing podcast audio from Freakonomics using `faster-whisper`</strong></font>

Name: Simon Reichel </br>
Semester: WS2025/2026

# Environment info

Like last time, I am going to supply the environment information for you to check if you run into compatibility issues. This time things are a bit more complicated for two reasons:
1. We are using two devices - a CPU and a GPU.
2. We are using a bunch of packages that depend on each other and need to be installed in very specific versions to work properly together.</br>
</br>
<strong>Environment information:</strong></br>
OS: Microsoft Windows 11 Pro (Developer Mode activated, `sudo` inactive) (x86)</br>
Version: 25H2</br>
Build: 26200.7171</br>
Package: Windows Feature Experience Pack 1000.26100.265.0</br>
Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]</br>
IPython: 8.37.0</br>
JupyterLab: 4.5.1</br></br>
CPU: Intel Core i5-12600KF with an x86 instruction set </br>
GPU: NVIDIA Blackwell RTX 5070Ti 16G</br></br>
If you run into a problem that does not seem to be connected to your environment feel free to report a bug to <a href=mailto:s.reichel@campus.lmu.de> the author</a>.
</br></br></br>

# What are we actually doing today?

We have multiple tasks planned for this notebook. Our main goal is to do audio transcription using `faster-whisper`, a ML-model developed by Klein et al. You can get more information about it on their Github [here](https://github.com/SYSTRAN/faster-whisper). </br>
In general, `faster-whisper` is a reimplementation of the original `whisper` model developed by OpenAI (available [here](https://github.com/openai/whisper)). </br>
`faster-whisper` is considered to use less memory while at the same time being faster than `whisper`. </br></br>

We will put performance to the test using different model sizes. We will both check how long it takes to transcribe our reference audio files and how accurate that transcription is. </br>
As reference audio we are going to use episodes from the Freakonomics podcast. We will get those files via RSS-Feed from Simplecast. </br></br>

We have considered the following ethical considerations:
1. <i>Is this approach legal?</i> </br>
   <strong>Answer:</strong> The exact details may vary depending on the legislation in your country. We are conforming to German law. First of all, accessing an RSS-Feed is legal. One could argue that this is what an RSS-Feed is designed for. Requests are anticipated server-side and pose no problem. Second, downloading content from an RSS-Feed can be legal or illegal depending on the intended use-cases. In general, podcasts are protected by copyright law. Downloading a podcast means making a copy of it which is an action that might require consent from the copyright owners. However, one could argue that by making a podcast available for download via RSS-Feed can be considered as consent to actually do this. Furthermore, the German copyright law (UrhG) makes certain exceptions for use of content in non-commercial research. Specifially, §60d UrhG allows us to copy protected content as long as it is used in non-commercial research only. §60d Abs. 3 UrhG also allows us to distribute and share copyright-protected content with other researchers as long as it serves research or verification purposes. We are therefore entitled to download and use the podcasts as intended. All calculations are being done on a local workstation which makes accidental leak of copyright-protected content unlikely.

2. <i>Machine learning needs a lot of energy and water. Is it ethical to use ML for a task like this?</i> </br>
    <strong>Answer:</strong> Again, this might be subject to your specific compute environment. We are using a local workstation, so we have full control over all of the mentioned aspects. First of all, we use a GPU from NVIDIA's Blackwell architecture, which is the most current architecture. It uses 5th. generation tensor cores and a 2nd. generation transformer engine. This GPU is arguably the most efficient piece of hardware possible for a task like this. Additionally, our workstation draws 100% of its power from renewable energy sources. Furthermore, 100% of the generated heat is recycled. We can therefore ensure a maximum of efficiency, far surpassing most datacenter efficiency claims. We are able to reduce environmental impact to a minimum. Under these circumstances, we consider the use of ML acceptable.

# Why is that relevant for Social Science?

# Actual implementation

## Pulling podcast audio files via RSS-Feed

First of all, we define our working directory. The `"` are important as we have a whitespace in this path.

In [1]:
cd "C:/Users/Simon/Documents/02_Master_Computational_Social_Science/1 Semester/datascraping/working_directory"

C:\Users\Simon\Documents\02_Master_Computational_Social_Science\1 Semester\datascraping\working_directory


C:\Users\Simon\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


To pull content from RSS-Feeds - including the URL's for our audio files - we only need Python base utilities. </br>
We are going to need: 
- `requests` to get content from URL's
- `os` to save files
- `re` to manipulate expressions
- `time` to implement sleep()-statements
- `xml.etree.ElementTree` to search/modify XML-documents
- `urllib.parse` to decode and parse URL's

So lets import all of them:

In [2]:
import requests
import os
import re
import time
import xml.etree.ElementTree as ET #save some typing work
from urllib.parse import urlparse #we only need this function

We want to save files to our mass storage device. However, it is possible that our file names contain illegal characters not recognized by our file system. We are therefore implementing a function to remove all illegal characters. Depending on your OS, you might need to add or remove characters. </br>
This functions swaps out all illegal characters with an `_`. 

In [3]:
def clean_filename(filename):
    return re.sub(r'[<>:"/\\|?*]', '_', filename)

Now we can define a function to pull the podcast audio files we are interested in. </br>
We need a RSS-Feed URL to pull from, which is this one for Freakonomics: https://feeds.simplecast.com/Y8lFbOT4. You can actually open this URL in your browser to see what the content looks like.</br>
We then fetch and parse the content from this URL and get the audio files to download. </br>
> <strong>Note:</strong> This definition is quite long, but inline comments have been added to make it (relatively) easy to understand.

In [4]:
def get_audiofiles(feed_url, amount, sleep_timer):

    #define list for filenames
    list_podcasts = []
    
    #fetch and parse
    response = requests.get(feed_url)
    root = ET.fromstring(response.content)

    #extract and download episode audio files

    #print info about amount of items for debugging
    #if this prints "Found items: 0", something is wrong
    items = root.findall(".//item")
    print("Found items:", len(items))
    
    #find all items from first up to "amount" 
    for item in root.findall(".//item")[:amount]:

        #find publication date; if unavailable fall back to "NA"
        date = item.findtext("pubDate", default = "NA")
        
        #find title; if unavailable fall back to "episode"
        title = item.findtext("title", default="episode")
       
        #find enclosure; return "None" if unavailable
        enclosure = item.find("enclosure")

        #check if enclosure returned "None"
        #Reason: .attrib() might crash if being fed "None"
        # also check if enclosure type is audio (see below)
        if enclosure is not None and enclosure.attrib.get(

            #get type = audio; if unavailable return empty
            "type","").startswith("audio"):

            #extract audio url (if it exists)
            audio_url = enclosure.attrib.get("url")
            
            # limit length of filename
            name = (title)[:80]
            
            #use clean_filename function from above
            #also add file type (.mp3) to filename
            filename = f"{clean_filename(name)}.mp3"

            #print filename
            print(f"Downloading: {filename}")
            
            #get audiofile
            #use "stream = True" to get it sequentially
            r = requests.get(audio_url, stream = True)
           
            #save file to mass storage device
            #use "wb" to save as binary
            with open(filename, "wb") as f:

                #chunk size is determined in bytes
                #therefore we read eight kilobytes per chunk
                #these are technically kibibyte and not kilobyte but that is not important right now
                #keep chunk size small for better efficiency
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            #confirm successful save
            print(f"{filename} was successfully saved")

            #create dictionary
            dict_podcast = {}

            #add title, date and filename to dictionary
            dict_podcast["title"] = title
            dict_podcast["date"] = date
            dict_podcast["filename"] = filename
            
            #append dictionary to list
            list_podcasts.append(dict_podcast)
            
            #sleep to avoid hammering the server
            time.sleep(sleep_timer)

    #return list for outside use
    return list_podcasts

The function above might seem overwhelming at first glance. To be fair: It does a lot of things. We now can:
- parse a RSS-Feed URL
- get a predetermined number of entries (called "amount")
- get metadata like titles and publication dates for each entry
- extract a filename and format it in such a way it can be saved as a file
- save said file
- write the metadata into a dictionary and append it to a list
- return the list to the outside
- let the function sleep for a predetermined time (called "sleep_timer")

And the best part? We can do all of this with base Python utilities. Not a single third-party package is needed for all of this. We can therefore ensure maximum compatibility across various Python environments.

</br></br>

Lets run our function and see if it works. We define the correct URL and want to get the first 15 entries. We also set our sleep-timer to ten seconds after each query. Ten seconds is basically an eternity for computers, but for 15 entries it does not really matter and we can be civil and polite with the server. </br>
A long sleep-timer (maybe even randomized - not implemented here) also reduces the risk of an HTTP-429 error. Although on RSS-Feeds the ratelimits are usually forgiving.

In [5]:
# RSS feed for Freakonomics Radio
RSS_URL = "https://feeds.simplecast.com/Y8lFbOT4"

#run function
list_podcasts = get_audiofiles(RSS_URL, #url to pull from
                               15, #get first 15 entries
                               10 #sleep for ten seconds after each run
                              )

Found items: 885
Downloading: 661. Can A.I. Save Your Life_.mp3
661. Can A.I. Save Your Life_.mp3 was successfully saved
Downloading: 660. The Wellness Industry Is Gigantic — and Mostly Wrong.mp3
660. The Wellness Industry Is Gigantic — and Mostly Wrong.mp3 was successfully saved
Downloading: Steve Levitt Quits His Podcast, Joins Ours.mp3
Steve Levitt Quits His Podcast, Joins Ours.mp3 was successfully saved
Downloading: 659. Can Marty Makary Fix the F.D.A._.mp3
659. Can Marty Makary Fix the F.D.A._.mp3 was successfully saved
Downloading: 658. This Is Your Brain on Supplements.mp3
658. This Is Your Brain on Supplements.mp3 was successfully saved
Downloading: Are Personal Finance Gurus Giving You Bad Advice_ (Update).mp3
Are Personal Finance Gurus Giving You Bad Advice_ (Update).mp3 was successfully saved
Downloading: Are You Ready for a Fresh Start_ (Update).mp3
Are You Ready for a Fresh Start_ (Update).mp3 was successfully saved
Downloading: Are the Rich Really Less Generous Than the P

In [59]:
#use pprint for better structured print
from pprint import pprint
pprint(list_podcasts)

[{'date': 'Fri, 30 Jan 2026 11:00:00 +0000',
  'filename': '661. Can A.I. Save Your Life_.mp3',
  'title': '661. Can A.I. Save Your Life?'},
 {'date': 'Fri, 23 Jan 2026 11:00:00 +0000',
  'filename': '660. The Wellness Industry Is Gigantic — and Mostly Wrong.mp3',
  'title': '660. The Wellness Industry Is Gigantic — and Mostly Wrong'},
 {'date': 'Wed, 21 Jan 2026 01:00:00 +0000',
  'filename': 'Steve Levitt Quits His Podcast, Joins Ours.mp3',
  'title': 'Steve Levitt Quits His Podcast, Joins Ours'},
 {'date': 'Fri, 16 Jan 2026 11:00:00 +0000',
  'filename': '659. Can Marty Makary Fix the F.D.A._.mp3',
  'title': '659. Can Marty Makary Fix the F.D.A.?'},
 {'date': 'Fri, 9 Jan 2026 11:00:00 +0000',
  'filename': '658. This Is Your Brain on Supplements.mp3',
  'title': '658. This Is Your Brain on Supplements'},
 {'date': 'Fri, 2 Jan 2026 11:00:00 +0000',
  'filename': 'Are Personal Finance Gurus Giving You Bad Advice_ (Update).mp3',
  'title': 'Are Personal Finance Gurus Giving You Bad Ad

We are going to save the list with the metadata on our mass storage device. All the files have been saved already.

In [ ]:
with open("list_podcasts.json", "w") as f:
    json.dump(list_podcasts, f)

And that is it. We just downloaded 15 podcasts via RSS-Feed. </br>
Note that for a large number this might not necessarily work. RSS-Feeds usually only go back a certain timeframe like a month or maybe two months. This specific RSS-Feed seems to go back over a decade and gives us almost 900 items. For our example, 15 episodes are more than enough, however. </br>
Next, we will take on the audio transcription. 
</br></br></br></br></br>

## Transcribing audio using `faster-whisper` and performance checks

In the following section, we will use `faster-whisper`, a supposedly more efficient reimplementation of the original OpenAI `whisper`. </br>
We will transcribe audio from the podcasts we just downloaded. To do so, we will use different models of `faster-whisper` and compare them against each other. </br>
We will look at key metrics like:
- Performance (measured in runtime)
- Ressource use
- Accuracy
- Efficiency (Performance per Watt)

All of these metrics will help us find out which model is right for which environment. Let's find out. 

</br></br></br>

### Setting up a machine learning environment

We will need several pieces of software and hardware to run `faster-whisper` as it is a ML-model. It is expected that you already have these requirements fulfilled in your environment:
- Git
- NVIDIA CUDA (version that is compatible with your CUDA device. This notebook uses CUDA 12.8 to ensure compatibility both with the NVIDIA Blackwell architecture as well as NVIDIA cuBLAS and NVIDIA cuDNN)
- NVIDIA cuBLAS for High Performance Computing
- NVIDIA cuDNN for High Performance Computing
- Python 3.10 (compatibility is not guaranteed on other versions)

As installation and setup might be different based on your specific environment, it is not further explained here. You will surely find a way ;)

We are going to need `torch` if we want to have GPU-acceleration. Acceleration is highly recommended, otherwise computations will take forever. </br>
Unfortunately, installing `torch` on Windows can be a huge pain. `torch` and all its subsets like `torchaudio` and `torchvision` need to be installed in very specific versions or they will not properly work together. </br>
> <strong>Note:</strong> In my experience, it is usually best to just wipe the entire `torch`-install and get a complete reinstall instead of upgrading. </br>

We are going to do that. Make sure you include `torchvision` in this process. We do not really need `torchvision` itself for this project, but `torchaudio` pulls some of its stuff from `torchvision` and might not work without it.

Use `pip uninstall` and then name the packages to be uninstalled. Use `!` to signal a shell command to Jupyter. Also do not forget to put `-y` at the end to confirm.

In [ ]:
!pip uninstall torch torchvision torchaudio -y

Now to reinstalling: </br>
We do not just tell `pip` to install any version of `torch` (often, `pip` will install the cpu-only version, which is useless to us). </br>
Instead, we exactly specify the version of `torch` and `torchaudio` we want to install via the URL to the pytorch website. This helps us to prevent some errors when initializing them lateron.

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp310-cp310-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cu128/torchvision-0.25.0%2Bcu128-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached https://download.pytorch.org/whl/cu128/torchaudio-2.10.0%2Bcu128-cp310-cp310-win_amd64.whl.metadata (7.1 kB)
Using cached https://download.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp310-cp310-win_amd64.whl (2867.4 MB)
Using cached https://download.pytorch.org/whl/cu128/torchvision-0.25.0%2Bcu128-cp310-cp310-win_amd64.whl (9.0 MB)
Using cached https://download.pytorch.org/whl/cu128/torchaudio-2.10.0%2Bcu128-cp310-cp310-win_amd64.whl (1.9 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   --------


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


First of all, let us check if `torch` is able to detect a CUDA-capable device. We expect `True` if you have a NVIDIA GPU built into your system.
> NVIDIA CUDA is a parallel computing platform and API that allows high performance parallel computing on GPU's. In principle it still relies on the legendary Brook framework by Ian Buck, however it has been refined a lot over the years. </br>
> CUDA allows us to speed up parallel computing by several orders of magnitude while being highly efficient, especially with current NVIDA GPU architectures.

In [1]:
import torch
torch.cuda.is_available()

True

Next, we can check what NVIDIA CUDA device is available.

In [3]:
!nvidia-smi

Sat Jan 31 20:52:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   45C    P8             14W /  300W |    1955MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

We can see that our reference system uses a NVIDIA Blackwell RTX 5070Ti with 16 Gigabytes of GDDR7 memory and a power target of 300 Watts. </br>
And just to be safe, we can also verify which version of `torch` and CUDA is running.

In [4]:
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

Torch version: 2.10.0+cu128
CUDA version: 12.8


Curiously, `torch` tells us we are using CUDA 12.8 while NVIDIA tells us we are using CUDA 13.1.

We have now verified:
- `torch` is able to detect a device
- This device is a NVIDIA Blackwell RTX 5070Ti
- We use `torch` 2.10.0
- We use CUDA 13.1

This might seem like a lot of verifying just for some packages we imported. But in machine learning nothing is "too much". Considering that we are going to do a ton of computations, we want to be sure everything is up and running as it should be. </br>
A wrong configuration might cost us a lot of additional compute time. </br></br>
Finally, we can set our CUDA-device as standard. We also define our CPU as fallback, should the CUDA-device be unavailable. Running computations on CPU is however not recommended, as it is much slower.

In [7]:
#set device to cuda if available
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

#check which device is being used
print("device being used:", device)

device being used: cuda


</br></br></br>

### Installing and running `faster-whisper`

Now that we have spend a lot of time setting up our machine learning environment correctly, we can finally get to install and use `faster-whisper`. </br>
We are installing the package via `pip`.

In [1]:
!pip install faster-whisper


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------- ----------- 0.8/1.1 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 5.5 MB/s  0:00:00
   ---------------------------------------- 0.0/18.6 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.6 MB 5.6 MB/s eta 0:00:04
   ----- ---------------------------------- 2.4/18.6 MB 6.1 MB/s eta 0:00:03
   ------- -------------------------------- 3.4/18.6 MB 6.1 MB/s eta 0:00:03
   --------- ------------------------------ 4.5/18.6 MB 5.7 MB/s eta 0:00:03
   ----------- ---------------------------- 5.5/18.6 MB 5.6 MB/s eta 0:00:03
   -------------- ------------------------- 6.6/18.6 MB 5.4 MB/s eta 0:00:03
   ---------------- ----------------------- 7.6/18.6 MB 5.4 MB/s eta 0:00:03
   ------------------ --------------------- 8.7/18.6 MB 5.4 MB/s eta 0:00:02
   -------------------- ------------------- 9.7/18.6 MB 5.4 MB/s eta 0:00:02
   ---------------

As always, we can get some information about this package by typing `pip show`. </br>
We can see all the dependencies of this package and also get a link to the authors Github.

In [2]:
!pip show faster-whisper

Name: faster-whisper
Version: 1.2.1
Summary: Faster Whisper transcription with CTranslate2
Home-page: https://github.com/SYSTRAN/faster-whisper
Author: Guillaume Klein
Author-email: 
License: MIT
Location: c:\users\simon\appdata\local\programs\python\python310\lib\site-packages
Requires: av, ctranslate2, huggingface-hub, onnxruntime, tokenizers, tqdm
Required-by: 


Now let's get started with the model. </br>
First we import the model and define all the model variants we are going to test today.

In [2]:
from faster_whisper import WhisperModel
import time

#largest model for best accuracy
model_large = "large-v3"

#largest model with simplified decoder for more speed
model_large_turbo = "large-v3-turbo"

#medium model for mix of speed and accuracy
model_medium = "medium"

#smallest model for best speed in low compute environments
model_tiny = "tiny"

We can set different types of precision. Basically, "precision" defines how many decimals are taken into consideration when calculations are being done. It is a bit more complicated than that as we are often using floating point numbers which are split into an exponent and a mantissa plus signage but we will not get into the details here. In simple terms, a higher number will mean better precision. </br>
We will define the classical FP16 precision for GPU-accelerated computing. we also define INT8 compute for CPU, altough we might not use it.
> <strong>Note:</strong> Remember that CPU compute is only defined as fallback if `torch` is unable to detect a CUDA-capable device. We specified this when we set up our environment. CPU usually does not support FP16 precision, but it might support INT8 and INT32 precision.

In [3]:
#set standard precision for GPU
fp16 = "float16"

#set reduced precision for CPU
cpu_int8 = "int8"

Now we are going to define a function to run our model and finally transcribe some data. </br>
The function expects five arguments as follows:
1. <strong>model_size:</strong> Expects model to use as defined above
2. <strong>precision:</strong> Expects precision to use as defined above
3. <strong>file:</strong> Expects the file to transcribe
4. <strong>quality:</strong> Expects an integer. Higher means better quality but reduced speed in transcribing
5. <strong>timestamps:</strong> Expects `True` (timestamps will be supplied) or `False` (timestamps will not be supplied)

In [4]:
def get_transcription(model_size, precision, file, quality, timestamps):

    #define list
    output = []
    
    #start time measurement for performance metrics
    start_time = time.perf_counter()
    
    model = WhisperModel(model_size, #choose model size

                         #set device
                         #we already set the device above
                         device = device, 

                         #choose precision like FP16 or INT8
                         compute_type = precision 
                        )
    
    segments, info = model.transcribe(
        
        #provide file path
        file,

        #set quality
        #higher numbers get better quality but lower speed
        beam_size = quality,

        #choose if timestamps should be supplied
        word_timestamps = timestamps 
    )
    
    #start iterating through segments
    
    #check if timestamps should be supplied
    if timestamps == True:
        for segment in segments:

            #print segment text and timestamp
            print(f"[{segment.start:.2f}s — {segment.end:.2f}s]: {segment.text}")

            #create dictionary
            dict_transcription = {}

            #add start, end and text to dictionary
            dict_transcription["start"] = segment.start
            dict_transcription["end"] = segment.end
            dict_transcription["text"] = segment.text
        
            #append dictionary to list
            output.append(dict_transcription)
    
    #if timestamps should not be supplied
    else:
        output = " ".join(segment.text for segment in segments)
    
    #end time measurement
    end_time = time.perf_counter()
    
    #print passed time
    print(f"This function was completed in {end_time - start_time:.6f} seconds")

    #return output for outside use
    return output


Now we have a function that does several things:
- Transcribes audio from provided file including timestamps
- lets us set model size, precision, quality and timestamps (if we want them)
- Measures its own runtime giving us a performance indicator

We want to get further performance metrics than just runtime. We also want to take a look at ressource  requirements. </br>
To do this, we use a calssic tool in the benchmarking field: HWINFO-64. </br>
This tool allows us to inspect a ton of sensor data from our system. Take a look at the images below where you can see all the information we are getting on our CPU and GPU.
</br></br>

![CPU_monitoring_HWINFO-64](https://raw.githubusercontent.com/47867/datascraping_2025-2026/main/CPU_monitoring_HWINFO.png)
![GPU](https://raw.githubusercontent.com/47867/datascraping_2025-2026/main/GPU_monitoring_HWINFO.png)



These images might seem overwhelming if you do not know what to look for. However if you <strong>do know</strong> what to look for, this sensor data can tell you a lot. </br>
We are going to use some of this data to approximate ressource use. 
</br></br>
> <strong>Note:</strong> This is only an approximation to ressource use as the monitoring itself takes some performance (probably around one percent). Also there are additional background processes which might have an impact on performance. So take these values with a grain of salt. </br>
> Real performance monitoring is much more complex than what we are doing today and has to account for a lot of factors like temperature, input-voltages, run-to-run variance, load-management, silicon-quality and many more.

### Transcribing our podcasts

We are almost done here. Lets take a look back at what we have achieved so far. </br>
We...
- ... defined a function to pull podcasts and executed it
- ... set up a high-performance machine learning environment
- ... set up `faster-whisper`
- ... defined a function to run `faster-whisper`
- ... set up performance measurement

Now, we can finally start transcribing and measure our performance across various models. </br>
We will start with the `tiny` model which is the smallest te set up a baseline.


In [25]:
tiny_transcription = []

for i in range(len(list_podcasts)):
    tiny_transcription.append(get_transcription(
        model_tiny,
        fp16,
        list_podcasts[i]["filename"],
        5,
        False
    )
                             )

This function was completed in 92.564415 seconds
This function was completed in 96.830416 seconds
This function was completed in 64.282420 seconds
This function was completed in 77.653959 seconds
This function was completed in 74.487054 seconds
This function was completed in 82.821976 seconds
This function was completed in 62.660185 seconds
This function was completed in 60.272495 seconds
This function was completed in 63.943810 seconds
This function was completed in 58.959409 seconds
This function was completed in 75.161151 seconds
This function was completed in 75.155202 seconds
This function was completed in 85.994262 seconds
This function was completed in 71.660449 seconds
This function was completed in 82.206156 seconds


Okay, our run with the `tiny` model is done. Lets save the result to a JSON on our mass storage device. 

In [27]:
import json

with open("tiny_transcription.json", "w") as f:
    json.dump(tiny_transcription, f)

Next up: The `medium` model.

In [31]:
medium_transcription = []

for i in range(len(list_podcasts)):
    medium_transcription.append(get_transcription(
        model_medium,
        fp16,
        list_podcasts[i]["filename"],
        5,
        False
    )
                             )

This function was completed in 297.898647 seconds
This function was completed in 342.628255 seconds
This function was completed in 242.049636 seconds
This function was completed in 259.632890 seconds
This function was completed in 257.471364 seconds
This function was completed in 303.846669 seconds
This function was completed in 209.665620 seconds
This function was completed in 192.447255 seconds
This function was completed in 210.373366 seconds
This function was completed in 206.968166 seconds
This function was completed in 256.113404 seconds
This function was completed in 261.265362 seconds
This function was completed in 287.019467 seconds
This function was completed in 244.134807 seconds
This function was completed in 274.528647 seconds


After completion, we again write all the content in a JSON file.

In [32]:
with open("medium_transcription.json", "w") as f:
    json.dump(medium_transcription, f)

Now, only the `large` models remain. There are two `large` models, a standrad one and a allegedly faster one called `turbo`. </br>
By using a simplified decoder, the `large-turbo` model claims faster performance by sacrificing some accuracy. We are going to put these claims to the test.

In [34]:
large_turbo_transcription = []

for i in range(len(list_podcasts)):
    large_turbo_transcription.append(get_transcription(
        model_large_turbo,
        fp16,
        list_podcasts[i]["filename"],
        5,
        False
    )
                             )

This function was completed in 116.087309 seconds
This function was completed in 109.758763 seconds
This function was completed in 89.126427 seconds
This function was completed in 184.303799 seconds
This function was completed in 83.278054 seconds
This function was completed in 116.322054 seconds
This function was completed in 83.418727 seconds
This function was completed in 79.705599 seconds
This function was completed in 113.691623 seconds
This function was completed in 78.614933 seconds
This function was completed in 144.273999 seconds
This function was completed in 110.609343 seconds
This function was completed in 107.701580 seconds
This function was completed in 89.706218 seconds
This function was completed in 91.690942 seconds


In [35]:
with open("large_turbo_transcription.json", "w") as f:
    json.dump(large_turbo_transcription, f)

In [38]:
large_transcription = []

for i in range(len(list_podcasts)):
    large_transcription.append(get_transcription(
        model_large,
        fp16,
        list_podcasts[i]["filename"],
        5,
        False
    )
                             )

This function was completed in 421.221626 seconds
This function was completed in 542.767554 seconds
This function was completed in 328.692195 seconds
This function was completed in 402.106266 seconds
This function was completed in 399.133849 seconds
This function was completed in 432.833018 seconds
This function was completed in 292.139069 seconds
This function was completed in 328.921933 seconds
This function was completed in 428.324944 seconds
This function was completed in 299.573883 seconds
This function was completed in 439.750274 seconds
This function was completed in 471.250958 seconds
This function was completed in 453.977419 seconds
This function was completed in 414.008552 seconds
This function was completed in 452.818094 seconds


In [40]:
with open("large_transcription.json", "w") as f:
    json.dump(large_transcription, f)

We have now successfully transcribed all of our 15 podcasts with the `tiny`, `medium`, `large-turbo` and `large` model. </br>
It took us this long with our NVIDIA Blackwell GPU:


    
model | time in seconds | time in minutes|
:------:|:-----------------:|:----------------:|
**tiny** | 1124 seconds | ca. 19 minutes|
**medium** | 3846 seconds | ca. 64 minutes |
**large-turbo** | 1598 seconds | ca. 27 minutes |
**large** | 6108 seconds | ca. 102 minutes |



It is important to note that we used our Tensor cores only to maximize efficiency. We could speed up calculations by using also our CUDA cores but we would lose efficiency. </br></br>
<strong>What are Tensor cores and CUDA cores you might ask?</strong> </br></br>
Here is the short version: Modern NVIDIA GPU's have three kinds of different cores (cores are basically the "execution units" of a processor). There are CUDA cores, Tensor cores and Raytracing cores. The last category is not really relevant for us right now (they can be used for complicated calculations considering the reflection and absorption of light when it hits a surface. These calculations are important in a range of fields). Tensor cores are execution units that are specifically designed for matrix multiplications. Matrix multiplications are operations that play a huge role in machine learning. And tensor cores are incredibly fast at calculating them. Cuda cores on the other hand are general-purpose cores that can do everything, just not as efficient. So cuda cores can do matrix multiplications as well, but they are much slower at it and need more power. To maximize efficiency, we only use tensor cores for matrix multiplications in this notebook even if we have to sacrifice some raw performance. </br>
> <strong>Note:</strong> Reality is of course much more complex than that. Modern GPU's are an engineering marvel but we will not go into the details here.

But what if we do not have a CUDA-capable device? This is quite possible if you use a laptop without a discrete graphics solution for example. </br>
So - just for fun - we are going to do some CPU compute. 
> <strong>Note:</strong> CPU-compute is not recommended for machine learning. It takes quite some time and might not support your model. CPU's often do not natively support FP16 compute and fallback to FP32 to emulate FP16 which is much slower and defeats the purpose of FP16. Depending on your CPU INT8 and INT32 are the most viable options. 

In [46]:
#change device to cpu
device = "cpu"

#run model for a subset only in INT8
cpu_transcription = []

for i in range(2):
    cpu_transcription.append(get_transcription(
        model_tiny,
        cpu_int8,
        list_podcasts[i]["filename"],
        5,
        False
    )
                             )

#reset device to cuda
device = "cuda"

This function was completed in 175.939783 seconds
This function was completed in 190.886484 seconds


This is the `tiny` model and it took us more than six minutes to transcribe just two episodes. On our GPU it took just half that time. </br>
And remember: This is not even FP16, it is INT8 which is a lower precision. </br>
Lets save the content to a JSON and move on to the performance evaluation.

In [51]:
with open("cpu_transcription.json", "w") as f:
    json.dump(cpu_transcription, f)

### Performance

In this section we will loke at different performance metrics. These include:
- raw performance
- ressource usage
- accuracy
- efficiency (performance per watt)

We already took a look at the raw compute times above. Surprisingly, the `large-turbo` model only takes eight minutes longer than the `tiny` model. Apparantly the description `turbo` really fits this model. But at what price does that speed come?

> At this point I originally planned to pull transcripts from the Freakonomics website to compare against our transcriptions. </br>
> Unfortunately however, altough their robots.txt does not forbid crawling, Freakonomics does not like bots on their page -> we get a HTTP-403 error which means we are not allowed on their page. </br>
> They employ several anti-bot measures like Captcha's and TLS-fingerprinting. </br></br>
> It is of course totally possible to circumvent all of these measures. However, some of the things we would need to do are not exactly legal in Germany.</br>
> According to §269 Strafgesetzbuch, we could face up to five years in jail for TLS-spoofing, which would be necessary to circumvent the TLS-fingerprinting. </br>
> Again, TLS-spoofing is not impossible, but I am not going to explain it here (it is not as easy as it appears to be as well). Maybe look somewhere else if you are interested in it. For purely theoretical reasons of course (no really, try to avoid committing a federal felony).

</br></br></br></br></br></br></br></br>
# To-Do: judge accuracy manually
</br></br></br></br></br></br></br></br>

Now lets take a look at the ressources our models required to run. We logged all the sensor data via HWINFO-64. </br>
HWINFO logs data every two seconds. So for the 3846 seconds it took us to run the `medium` model, we have around 1900 datapoints. </br>
We can then calculate the mean over all of those datapoints and get an approximation on the ressource usage. </br></br>
If we collect all the sensor data from HWINFO, we get the following table:

model | avg. VRAM usage | avg. gpu power consumption | avg. cpu power consumption | runtime | total power consumption |
:----:|:---------------:|:------------------:|:------------------:|:------------:|:------------------:|
tiny | 0.243 GB | 58.492 W | --- | 1124 s | 18.263 Wh |
medium | 2.194 GB | 81.841 W | --- | 3846 s | 87.433 Wh |
large-turbo | 2.341 GB | 84.159 W | --- | 1598 s | 37.357 Wh |
large | 4.002 GB | 92.266 W | --- | 6108 s | 156.545 Wh
tiny (CPU INT8) | --- | ---| 55.844 W | 367 s* | 5.693 Wh* |
*only two episodes transcribed

Several things are immediately noticeable: Even the `large` model only needs around four gigabytes of VRAM which is essentially nothing considering that we often need hundreds of gigabytes for LLM's. You could easily load four gigabytes into memory even on small GPU's and also on systems without discrete GPU where it instead gets loaded into DRAM instead of VRAM. </br>
Also, power draw on the GPU seems to be quite low. The power target for this GPU is set to 300 Watts, but it barely surpasses 90 Watts on the `large` model. This is because we only use the tensor cores and no cuda cores which are basically at idle the entire time. </br>
Additionally, the `large` model only needs around 157 Watthours (= 0.157 kWh) round-trip for all the calculations which is not a lot in a desktop environment. In a laptop environment, things look differently. Most laptops have a battery smaller than a 100 Watthours (otherwise they would not be allowed on airplanes). So you would either have to plug it into the wall or use the `large-turbo` model, which could theoretically run on a laptop battery. However, this heavily depends on the efficiency of your hardware.

Now lets also calculate efficiency. We define efficiency as transcribed seconds per Watthour. So lets first get the length of our audio files. </br>
We are going to get that info via `mutagen`.

In [4]:
!pip install -U mutagen


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Afterwards, we add the length of each podcast to our list we defined at the beginning of this notebook.

In [13]:
from mutagen.mp3 import MP3

#get length for each episode
for i in range(len(list_podcasts)):
    
    #get audio file
    audio = MP3(list_podcasts[i]["filename"])
    
    #get length in seconds as a float
    length_seconds = audio.info.length

    #append to dictionary/list
    list_podcasts[i]["length"] = length_seconds 

#print to verify
print(list_podcasts[0])

{'title': '661. Can A.I. Save Your Life?', 'date': 'Fri, 30 Jan 2026 11:00:00 +0000', 'filename': '661. Can A.I. Save Your Life_.mp3', 'length': 3611.4285625}


We are now able to sum up all of these values to get the total length of all podcasts combined.

In [32]:
#sum up length
length_combined = sum(item["length"] for item in list_podcasts)

#print to verify
print(f"seconds:", length_combined, f", minutes:", length_combined//60)

seconds: 48601.4689375 , minutes: 810.0


Now lets see how many seconds or minutes we can transcribe with each model per Watthour.

model | total power consumption | efficiency (seconds)* | efficiency (minutes)*
----|:-----------------------:|:-----------:|:-------------------:|
tiny | 18.263 Wh | 2661.271 s | 44.353 min |
medium | 87.433 Wh | 555.868 s | 9.264 min |
large-turbo | 37.357 Wh | 1300.992 s | 21.683 min |
large | 156.545 Wh | 310.464 s | 5.174 min |
tiny (CPU INT8)** | 5.693 Wh | 1324.637 s | 22.077 min |
*Transcribed seconds or minutes per Watthour
**Only two episodes transcribed

Out of all these, the `tiny` model is the most efficient, giving us 2661 seconds or 44 minutes of transcription per Watthour of energy used. The `large-turbo` model is also respectable, giving us 1300 seconds or 21 minutes of transcribed audio for each Watthour of energy used. </br>
The CPU also gives us around 1324 seconds or 22 minutes of transcription per Watthour. However, this is in INT8 instead of FP16 and on the `tiny` model. So the GPU is much more efficient.

We can also check how many seconds or minutes of audio each model transcribed per second of runtime.

model | total transcription runtime | efficiency (seconds)* | efficiency (minutes)*
----|:-----------------------:|:-----------:|:-------------------:|
tiny | 1124 s | 43.240 s | 0.721 min |
medium | 3846 s | 12.637 s | 0.211 min |
large-turbo | 1598 s | 30.414 s | 0.507 min |
large | 6108 s | 7.957 s | 0.133 min |
tiny (CPU INT8)** | 367 s | 20.548 s | 0.342 min |
*Transcribed seconds or minutes per second of runtime
**Only two episodes transcribed

Here our GPU-compute can show what it is capable of. Our CPU can only transcribe 21 seconds of audio per second of runtime on the `tiny` model in INT8. Our GPU can do 43 seconds of transcribing per second of runtime in the higher FP16 precision. So our GPU is much faster than the CPU, even though we are only using the efficient tensor cores which make up less than 20% of the GPU. </br>
If we were to use all cores including the less efficient cuda cores, we could stomp the CPU into the ground with raw performance but would lose energy efficiency. </br>
Also, the NVIDIA Blackwell RTX 5070Ti we are using here is only a desktop card. If we were to use the much more powerful datacenter cards like a B200 192GB or a H100 80GB we could increase performance by alot by running multiple instances of `faster-whisper` at the same time.

# Final conclusion

The obvious things first: Our NVIDIA GPU is not only faster than CPU-compute, it is also more efficient. In both efficiency ratings, the GPU is roughly twice as efficient as the CPU. If we assume, that decreasing precision on our GPU from FP16 to INT8 will probably give us roughly a 100% performance gain, we can assume the GPU to be four times as efficient as the CPU under equal conditions. This just shows again how efficient the Blackwell tensor cores really are. </br></br>
We can also see that running these models is possible in a range of environments. It does not really matter if we have a high performance desktop class workstation or a laptop without discrete graphics solution. Both of these machines should handle `faster-whisper` quite well. Most models can theoretically run on a laptop battery without need to plug into the wall, which is frankly incredible given that we speak about ML (even though these are very small models).

</br></br></br></br></br></br></br></br></br></br></br></br></br></br></br>
<font size = 7>
    
# ToDo:
# Judge accuracy manually (see chapter above)
# Relevance for social science
# Write final conclusion


</font></br></br></br></br></br></br></br></br></br></br></br></br></br></br></br>

# Experimental: One ring... ahhem... function to find them all and bind them

Lets try to build a function that does "everything" from start to finish. Note  that this function is explicitly marked as <strong>experimental</strong>. It is only tested in a very limited environment, it has two dozen dependencies which all rely on each other and need to be installed in very specific versions. And even Python itself needs to run in specific versions. It must not be to new, otherwise some CUDA dependencies might fail. But it must also not be too old for other dependencies to properly work. </br>
We are able to build a fragile environment here, but it could be difficult to replicate it. A container would be perfect for this, but I am probably not giong to build one as it is kind of out of scope for this assignment.</br></br>
For now: Consider this a highly volatile function that might very well crash your kernel, so save anything important before you run it.</br></br>



Functions expects the following arguments:

argument | class | values | default |description |
---------|:-----:|------|:-----------------:|--------|
feed_url | `string` | | | Define URL to retrieve from |
amount | `integer` | | 1 | Define amount to be retrieved and transcribed (Heavy hit on performance!) |
sleep_timer | `integer` or `float` | | 5 | Define sleep time after each run, can be left empty in most cases |
model_size | `string` | "tiny", "medium", "large-v3-turbo", large-v3" and other models | "large-v3-turbo"| Define model to use (Heavy hit on performance for higher models!) |
precision | `string` | "float16", "int8", "int32", "int8_float16" | "float16" | Define precision for ML |
quality | `integer` | | 5 | Higher means better quality (Medium hit on performance!) |
timestamps | `boolean` | `True`, `False` | `False` | Define if timestamps should be supplied |

In [5]:
import requests
import os
import re
import time
import xml.etree.ElementTree as ET #save some typing work
from urllib.parse import urlparse #we only need this function
import torch
from faster_whisper import WhisperModel
import time
import json


def the_lord_of_the_functions(feed_url, 
                              amount = 1, 
                              sleep_timer = 5, 
                              model_size = "large-v3-turbo", 
                              precision = "float16", 
                              quality = 5, 
                              timestamps = False
                             ):
    
    def clean_filename(filename):
        return re.sub(r'[<>:"/\\|?*]', '_', filename)
    
    #start time measurement for performance overview
    start_time = time.perf_counter()

    #set device to cuda if available
    if torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
    
    #define list for filenames
    list_podcasts = []
    
    #fetch and parse
    response = requests.get(feed_url)
    root = ET.fromstring(response.content)

    #extract and download episode audio files

    #print info about amount of items for debugging
    #if this prints "Found items: 0", something is wrong
    items = root.findall(".//item")
    print("Found items:", len(items))
    
    #find all items from first up to "amount" 
    for item in root.findall(".//item")[:amount]:

        #find publication date; if unavailable fall back to "NA"
        date = item.findtext("pubDate", default = "NA")
        
        #find title; if unavailable fall back to "episode"
        title = item.findtext("title", default="episode")
       
        #find enclosure; return "None" if unavailable
        enclosure = item.find("enclosure")

        #check if enclosure returned "None"
        #Reason: .attrib() might crash if being fed "None"
        # also check if enclosure type is audio (see below)
        if enclosure is not None and enclosure.attrib.get(

            #get type = audio; if unavailable return empty
            "type","").startswith("audio"):

            #extract audio url (if it exists)
            audio_url = enclosure.attrib.get("url")
            
            # limit length of filename
            name = (title)[:80]
            
            #use clean_filename function from above
            #also add file type (.mp3) to filename
            filename = f"{clean_filename(name)}.mp3"

            #print filename
            print(f"Downloading: {filename}")
            
            #get audiofile
            #use "stream = True" to get it sequentially
            r = requests.get(audio_url, stream = True)
           
            #save file to mass storage device
            #use "wb" to save as binary
            with open(filename, "wb") as f:

                #chunk size is determined in bytes
                #therefore we read eight kilobytes per chunk
                #these are technically kibibyte and not kilobyte but that is not important right now
                #keep chunk size small for better efficiency
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            #confirm successful save
            print(f"{filename} was successfully saved")

            #create dictionary
            dict_podcast = {}

            #add title, date and filename to dictionary
            dict_podcast["title"] = title
            dict_podcast["date"] = date
            dict_podcast["filename"] = filename
            
            #append dictionary to list
            list_podcasts.append(dict_podcast)
            
            #sleep to avoid hammering the server
            time.sleep(sleep_timer)

    
            #define list
            output = []
    
            model = WhisperModel(model_size, #choose model size

                                 #set device
                                 #we already set the device above
                                 device = device, 

                                 #choose precision like FP16 or INT8
                                 compute_type = precision 
                                )
    
            segments, info = model.transcribe(
        
                #provide file path
                filename,

                #set quality
                #higher numbers get better quality but lower speed
                beam_size = quality,

                #choose if timestamps should be supplied
                word_timestamps = timestamps 
            )
    
            #start iterating through segments
    
            #check if timestamps should be supplied
            if timestamps == True:
                for segment in segments:

                    #print segment text and timestamp
                    print(f"[{segment.start:.2f}s — {segment.end:.2f}s]: {segment.text}")

                    #create dictionary
                    dict_transcription = {}

                    #add start, end and text to dictionary
                    dict_transcription["start"] = segment.start
                    dict_transcription["end"] = segment.end
                    dict_transcription["text"] = segment.text
        
                     #append dictionary to list
                    output.append(dict_transcription)
    
            #if timestamps should not be supplied
            else:
                output = " ".join(segment.text for segment in segments)

            #safe to file
            with open(f"{clean_filename(name)}.json", "w") as f:
                json.dump(output, f)

            #sleep if function is to fast
            #unlikely to be needed here
            #lets specify it anayway
            time.sleep(sleep_timer)
                
    #end time measurement
    end_time = time.perf_counter()
    
    #print passed time
    print(f"This function was completed in {end_time - start_time:.6f} seconds")

Lets try it out with all the default values:

In [6]:
the_lord_of_the_functions("https://feeds.simplecast.com/Y8lFbOT4")

Found items: 886
Downloading: Why Don’t Running Backs Get Paid Anymore_ (Update).mp3
Why Don’t Running Backs Get Paid Anymore_ (Update).mp3 was successfully saved
This function was completed in 79.934130 seconds


<font size = 4>
    
<strong>Final conclusion: This function is obviously a bit silly, but it actually works.</strong> </br></br>

</font>

That is it for today. Lets give our poor system a bit of rest. Have fun running everything yourself and take care.